In [ ]:
%sql

--Cria catalog e schema
CREATE CATALOG IF NOT EXISTS lh_nautical;
CREATE SCHEMA IF NOT EXISTS lh_nautical.bronze;

In [ ]:
# Isolando a infraestrutura de conexão
from databricks.sdk.runtime import dbutils, spark 

db_user = dbutils.secrets.get(scope="sqlserver", key="username")
db_password = dbutils.secrets.get(scope="sqlserver", key="password")
db_host = dbutils.secrets.get(scope="sqlserver", key="host")
db_port = dbutils.secrets.get(scope="sqlserver", key="port")
db_database = dbutils.secrets.get(scope="sqlserver", key="db")

jdbc_url = f"jdbc:sqlserver://{db_host}:{db_port};databaseName={db_database}"

In [ ]:
import json

# 1. Carrega a lista de tabelas para ingestão integral na Bronze 
with open('src/tabelas_config.json', 'r', encoding='utf-8') as f:
    lista_tabelas = json.load(f)

# 2. Carrega a política de mascaramento dinâmica externa (quais colunas proteger)
with open('src/governanca_config.json', 'r', encoding='utf-8') as f:
    politica_governanca = json.load(f)

print("--- [CONFIGURAÇÃO DE INFRAESTRUTURA DE GOVERNANÇA] ---")

# 3. Cria Função de Mascaramento Dinâmico no Unity Catalog
# Esta regra avalia os novos grupos e só libera dados sensíveis para 'admins' e 'engenharia_dados'
spark.sql("""
CREATE OR REPLACE FUNCTION lh_nautical.bronze.regra_mascaramento(valor STRING)
RETURN CASE
  WHEN is_account_group_member('admins') THEN valor
  WHEN is_account_group_member('engenharia_dados') THEN valor
  ELSE '████████ (RESTRITO LGPD)'
END;
""")
print("[OK] Função de mascaramento dinâmico 'regra_mascaramento' injetada no catálogo.")

print(f"\nIniciando ingestão integral para as {len(lista_tabelas)} tabelas do ecossistema.")

# 4. Loop de Processamento Dinâmico (Extração Limpa + Vínculo de Máscaras)
for tabela in lista_tabelas:
    print(f"\n[JDBC] Extraindo tabela: {tabela}")
    
    jdbc_options = {
        "url":                    jdbc_url,
        "dbtable":                tabela,
        "user":                   db_user,
        "password":               db_password,
        "driver":                 "com.microsoft.sqlserver.jdbc.SQLServerDriver",
        "encrypt":                "true",
        "trustServerCertificate": "true",
        "loginTimeout":           "30",
        "fetchsize":              "1000",
    }
    
    # Leitura da tabela original na íntegra do SQL Server
    df_temp = spark.read.format("jdbc").options(**jdbc_options).load()
    
    # Persistência limpa na camada Bronze do Unity Catalog
    output_table = f"lh_nautical.bronze.{tabela}"
    df_temp.write.format("delta").mode("overwrite").saveAsTable(output_table)
    print(f"  └─► Tabela Delta base persistida com sucesso.")
    
    # Aplica a máscara dinâmica diretamente na coluna da tabela através do catálogo
    if tabela in politica_governanca:
        for coluna_sensivel in politica_governanca[tabela]:
            if coluna_sensivel in df_temp.columns:
                print(f"  └─► [UNITY CATALOG] Vinculando Máscara Dinâmica à coluna: '{coluna_sensivel}'")
                spark.sql(f"""
                    ALTER TABLE lh_nautical.bronze.{tabela}
                    ALTER COLUMN {coluna_sensivel} SET MASK lh_nautical.bronze.regra_mascaramento
                """)

print("\n--- [APLICAÇÃO DE PRIVILÉGIOS E SEGURANÇA RBAC] ---")

# 5. Distribuição de permissões no Schema Bronze para os grupos de atuação
print("[GRANT] Concedendo privilégio de APENAS LEITURA (SELECT) para o time de clientes (stakeholders)...")
spark.sql("GRANT SELECT ON SCHEMA lh_nautical.bronze TO `stakeholders`;")

print("[GRANT] Concedendo privilégios de LEITURA E ESCRITA (SELECT, MODIFY) para o time de engenharia_dados...")
spark.sql("GRANT SELECT, MODIFY ON SCHEMA lh_nautical.bronze TO `engenharia_dados`;")

print("\n[SUCESSO INTEGRAL] Pipeline finalizado, máscaras ativas e acessos distribuídos por área de atuação no projeto.")